# Timing Suite Loss Curves

Automatically loads the newest `timing_suite_mark2_*` run with `loss_history_all.csv` and plots loss curves for every experiment in that run.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

def find_repo_root(start=None):
    start = Path.cwd() if start is None else start
    for path in [start, *start.parents]:
        if (path / "experiments" / "timed_comparison").exists() and (path / "SAE.py").exists():
            return path
        if (path / "wdl_repo" / "experiments" / "timed_comparison").exists():
            return path / "wdl_repo"
    raise RuntimeError("Could not find repo root")

REPO_ROOT = find_repo_root()
RESULTS_DIR = REPO_ROOT / "experiments" / "results"

candidates = sorted(
    [p for p in RESULTS_DIR.glob("timing_suite_mark2_*") if (p / "loss_history_all.csv").exists()],
    key=lambda p: p.stat().st_mtime,
)
if not candidates:
    raise FileNotFoundError(f"No timing_suite_mark2_* runs with loss_history_all.csv found under {RESULTS_DIR}")

RUN_DIR = candidates[-1]
histories = pd.DataFrame()

print(f"Using RUN_DIR: {RUN_DIR}")
table_path = RUN_DIR / "timing_table_mark2.csv"
if table_path.exists():
    timing_df = pd.read_csv(table_path)
    display(timing_df)


In [ ]:
loss_history_path = RUN_DIR / "loss_history_all.csv"

def plot_loss_lines_matplotlib(plot_df, *, label_col, title="Loss curves"):
    plot_df = plot_df.dropna(subset=["elapsed_seconds", "loss"]).copy()
    if plot_df.empty:
        print("No plottable loss rows found.")
        return

    facet_col = "experiment" if "experiment" in plot_df.columns else None
    facets = list(plot_df[facet_col].dropna().unique()) if facet_col else [None]

    fig, axes = plt.subplots(
        1,
        len(facets),
        figsize=(6 * len(facets), 4),
        squeeze=False,
        sharex=False,
        sharey=False,
    )

    for ax, facet in zip(axes.flat, facets):
        if facet_col:
            sub = plot_df[plot_df[facet_col] == facet].copy()
            ax.set_title(str(facet))
        else:
            sub = plot_df.copy()
            ax.set_title(title)

        group_cols = [label_col]
        if "sample_size" in sub.columns:
            group_cols.append("sample_size")
        if "epsilon" in sub.columns:
            group_cols.append("epsilon")
        if "gamma" in sub.columns:
            group_cols.append("gamma")

        for keys, group in sub.groupby(group_cols, dropna=False):
            if not isinstance(keys, tuple):
                keys = (keys,)
            label = ", ".join(
                f"{col}={val}" for col, val in zip(group_cols, keys) if pd.notna(val)
            )
            group = group.sort_values("elapsed_seconds")
            ax.plot(
                group["elapsed_seconds"],
                group["loss"],
                marker="o",
                linewidth=1.5,
                markersize=4,
                label=label,
            )

        ax.set_xlabel("Elapsed seconds")
        ax.set_ylabel("Loss")
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

if loss_history_path.exists():
    loss_df = pd.read_csv(loss_history_path)
    display(loss_df.head())
    plot_loss_lines_matplotlib(loss_df, label_col="method")

elif not histories.empty:
    loss_cols = [c for c in ["train_loss", "loss", "objective", "recon_loss"] if c in histories.columns]
    if loss_cols and "elapsed_seconds" in histories.columns:
        loss_col = loss_cols[0]
        hist_plot = histories.dropna(subset=["elapsed_seconds", loss_col]).copy()
        hist_plot = hist_plot.rename(columns={loss_col: "loss"})
        label_col = "run_name" if "run_name" in hist_plot.columns else "history_file"
        display(hist_plot.head())
        plot_loss_lines_matplotlib(hist_plot, label_col=label_col)
    else:
        print("Histories were found, but no recognizable loss column was present.")
else:
    print("No JSONL histories or loss_history_all.csv found for this run.")
